# 06 Time-Frequency Consistency Pretraining

This notebook trains the Time-Frequency Consistency (TFC) encoder used in the proposed semi-supervised survival modelling framework.

The purpose of this notebook is to learn degradation-related sequence representations from vehicle operational histories before downstream survival fine-tuning.

The TFC approach uses two complementary views of the same vehicle history:

1. a time-domain representation based on cumulative temporal inputs;
2. a frequency-domain representation based on transformed differenced sequences.

The learned encoder weights are later used to initialize the downstream survival model.

## Inputs

- prepared sequence datasets from the sequence preparation stage;
- prefix-based sequence samples for SSL/TFC pretraining;
- time-domain and frequency-domain representations;

## Outputs

- pretrained TFC encoder checkpoint;
- training and validation loss history;

In [ ]:
# Set project paths and enable imports from src and models directories

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"
DATA_DIR = PROJECT_ROOT / "data" / "processed" / "sequences"
MODELS_DIR = PROJECT_ROOT / "models"

for path in [SRC_DIR, DATA_DIR, MODELS_DIR]:
    if str(path) not in sys.path:
        sys.path.append(str(path))

print(f"Notebook location: {Path.cwd()}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Source directory: {SRC_DIR}")
print(f"Models directory: {MODELS_DIR}")
print(f"Sequences directory: {DATA_DIR}")

In [ ]:
# Set random seed for reproducibility

import random
import numpy as np
import torch

RANDOM_STATE = 42

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)

In [ ]:
# check runtime GPU
import torch

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# load data
import pickle

with open(DATA_DIR / "train_sequences_full.pkl", "rb") as f:
    train_seq = pickle.load(f)

with open(DATA_DIR / "val_sequences_full.pkl", "rb") as f:
    val_seq = pickle.load(f)

print(len(train_seq), len(val_seq))

In [ ]:
# Inspect the first loaded sequence
first_vehicle = next(iter(train_seq))
sample_seq = train_seq[first_vehicle]

print("Vehicle id:", first_vehicle)
print("Keys:", sample_seq.keys())
print("Numerical shape:", sample_seq["numerical_sequence"].shape)
print("Histogram shape:", sample_seq["histogram_sequence"].shape)
print("Sequence length:", sample_seq["sequence_length"])

In [ ]:
from ssl_dataset import TFCPrefixDataset, tfc_prefix_collate_fn

print("Imports successful.")

In [ ]:
from tfc_preprocessing import build_tfc_prefixes_for_all_sequences, validate_tfc_prefix_dict, summarize_tfc_prefix_dict
train_prefix_dict = build_tfc_prefixes_for_all_sequences(
    sequence_dict=train_seq,
    proportions=(0.3, 0.6, 0.9),
    min_prefix_len=5,
    strict_prefix=True,
)

val_prefix_dict = build_tfc_prefixes_for_all_sequences(
    sequence_dict=val_seq,
    proportions=(0.3, 0.6, 0.9),
    min_prefix_len=5,
    strict_prefix=True,
)

validate_tfc_prefix_dict(train_prefix_dict, require_joint_sequences=False, require_scaled_time_gaps=False)
validate_tfc_prefix_dict(val_prefix_dict, require_joint_sequences=False, require_scaled_time_gaps=False)

print(summarize_tfc_prefix_dict(train_prefix_dict))
print(summarize_tfc_prefix_dict(val_prefix_dict))

In [ ]:
# instantiate dataset objects
train_dataset_ssl = TFCPrefixDataset(
    prefix_dict=train_prefix_dict,
    use_scaled_time_gaps=False
)

val_dataset_ssl = TFCPrefixDataset(
    prefix_dict=val_prefix_dict,
    use_scaled_time_gaps=False
)

In [ ]:
# inspect one sample
sample = train_dataset_ssl[0]

print("Keys:", sample.keys())
print("Prefix ID:", sample["prefix_id"])
print("Vehicle ID:", sample["vehicle_id"])
print("Sequence length:", sample["sequence_length"])
print("x_time shape:", sample["x_time"].shape)
print("x_freq shape:", sample["x_freq"].shape)
print("time_gaps shape:", sample["time_gaps"].shape)
print("time_steps shape:", sample["time_steps"].shape)

In [ ]:
# create the dataloader
from torch.utils.data import DataLoader

train_loader_tfc = DataLoader(
    train_dataset_ssl,
    batch_size=128,
    shuffle=True,
    collate_fn=tfc_prefix_collate_fn,
)

val_loader_tfc = DataLoader(
    val_dataset_ssl,
    batch_size=128,
    shuffle=False,
    collate_fn=tfc_prefix_collate_fn,
)

In [ ]:
# verify one batch
batch = next(iter(train_loader_tfc))

print("Batch keys:", batch.keys())
print("x_time shape:", batch["x_time"].shape)
print("x_freq shape:", batch["x_freq"].shape)
print("time_gaps shape:", batch["time_gaps"].shape)
print("time_steps shape:", batch["time_steps"].shape)
print("padding_mask shape:", batch["padding_mask"].shape)
print("sequence_lengths shape:", batch["sequence_lengths"].shape)
print("First 5 sequence lengths:", batch["sequence_lengths"][:5])

In [ ]:
def inspect_tfc_batch(batch: dict) -> None:
    print("prefix_ids:", batch["prefix_ids"][:3])
    print("vehicle_ids:", batch["vehicle_ids"][:3])
    print("x_time dtype:", batch["x_time"].dtype)
    print("x_freq dtype:", batch["x_freq"].dtype)
    print("time_gaps dtype:", batch["time_gaps"].dtype)
    print("padding_mask dtype:", batch["padding_mask"].dtype)
    print("x_time shape:", tuple(batch["x_time"].shape))
    print("x_freq shape:", tuple(batch["x_freq"].shape))
    print("time_gaps shape:", tuple(batch["time_gaps"].shape))
    print("padding_mask shape:", tuple(batch["padding_mask"].shape))
    print("min sequence length:", int(batch["sequence_lengths"].min()))
    print("max sequence length:", int(batch["sequence_lengths"].max()))

In [ ]:
inspect_tfc_batch(batch)

In [ ]:
import torch
import torch.nn.functional as F


"""
Losses and augmentations for TFC-style self-supervised pretraining.

This module provides:
- weak augmentations for the time branch
- weak augmentations for the frequency branch
- NT-Xent contrastive loss
- time-frequency consistency loss
- a wrapper that combines the losses into the TFC objective

Design principles
-----------------
- Augmentations are intentionally weak to avoid distorting the degradation
  structure of the operational trajectories.
- The time branch uses realistic perturbations in the sequence domain.
- The frequency branch uses light perturbations on the FFT magnitude space.
- The final loss combines:
    * time contrastive loss
    * frequency contrastive loss
    * time-frequency consistency loss
"""


def weak_time_augmentation(
    x: torch.Tensor,
    padding_mask: torch.Tensor,
    noise_std: float = 0.01,
    feature_dropout_prob: float = 0.05,
) -> torch.Tensor:
    """
    Apply weak augmentation to the time-domain sequence.

    Parameters
    ----------
    x : torch.Tensor
        Input tensor of shape (B, T, D).
    padding_mask : torch.Tensor
        Boolean mask of shape (B, T), where True indicates valid positions.
    noise_std : float, default=0.01
        Standard deviation of additive Gaussian noise.
    feature_dropout_prob : float, default=0.05
        Probability of dropping each feature value at valid positions.

    Returns
    -------
    torch.Tensor
        Augmented tensor of shape (B, T, D).
    """
    valid_mask = padding_mask.unsqueeze(-1).float()

    # Add small noise only on valid positions
    noise = torch.randn_like(x) * noise_std
    x_aug = x + noise * valid_mask

    # Apply small feature-wise dropout only on valid positions
    keep_mask = (
        torch.rand_like(x_aug) > feature_dropout_prob
    ).float()

    x_aug = x_aug * (keep_mask * valid_mask + (1.0 - valid_mask))
    return x_aug


def weak_frequency_augmentation(
    fft_mag: torch.Tensor,
    dropout_prob: float = 0.05,
    noise_std: float = 0.01,
) -> torch.Tensor:
    """
    Apply weak augmentation to the FFT magnitude representation.

    Parameters
    ----------
    fft_mag : torch.Tensor
        Frequency magnitude tensor of shape (B, T_f, D).
    dropout_prob : float, default=0.05
        Probability of dropping each value.
    noise_std : float, default=0.01
        Standard deviation of additive Gaussian noise.

    Returns
    -------
    torch.Tensor
        Augmented FFT magnitude tensor of shape (B, T_f, D).
    """
    noise = torch.randn_like(fft_mag) * noise_std
    fft_aug = fft_mag + noise

    keep_mask = (torch.rand_like(fft_aug) > dropout_prob).float()
    fft_aug = fft_aug * keep_mask

    return fft_aug


def nt_xent_loss(
    z1: torch.Tensor,
    z2: torch.Tensor,
    temperature: float = 0.2,
    eps: float = 1e-8,
) -> torch.Tensor:
    """
    Compute NT-Xent contrastive loss between two batches of projections.

    Parameters
    ----------
    z1 : torch.Tensor
        Projection tensor of shape (B, D).
    z2 : torch.Tensor
        Projection tensor of shape (B, D).
    temperature : float, default=0.2
        Temperature parameter.
    eps : float, default=1e-8
        Small constant for numerical stability.

    Returns
    -------
    torch.Tensor
        Scalar contrastive loss.
    """
    if z1.shape != z2.shape:
        raise ValueError("z1 and z2 must have the same shape.")

    batch_size = z1.size(0)
    if batch_size < 2:
        raise ValueError("NT-Xent requires batch_size >= 2.")

    z1 = F.normalize(z1, dim=1, eps=eps)
    z2 = F.normalize(z2, dim=1, eps=eps)

    representations = torch.cat([z1, z2], dim=0)  # (2B, D)
    similarity = torch.matmul(representations, representations.T) / temperature

    # Mask self-similarity
    diagonal_mask = torch.eye(2 * batch_size, device=z1.device, dtype=torch.bool)
    similarity = similarity.masked_fill(diagonal_mask, float("-inf"))

    # Positive pairs:
    # first half matches second half, second half matches first half
    positive_indices = torch.cat([
        torch.arange(batch_size, 2 * batch_size, device=z1.device),
        torch.arange(0, batch_size, device=z1.device),
    ])

    loss = F.cross_entropy(similarity, positive_indices)
    return loss


def time_frequency_consistency_loss(
    time_proj: torch.Tensor,
    freq_proj: torch.Tensor,
    eps: float = 1e-8,
) -> torch.Tensor:
    """
    Encourage time and frequency projections of the same sample to be close.

    Parameters
    ----------
    time_proj : torch.Tensor
        Time-branch projections of shape (B, D).
    freq_proj : torch.Tensor
        Frequency-branch projections of shape (B, D).
    eps : float, default=1e-8
        Small constant for numerical stability.

    Returns
    -------
    torch.Tensor
        Scalar consistency loss.
    """
    if time_proj.shape != freq_proj.shape:
        raise ValueError("time_proj and freq_proj must have the same shape.")

    time_proj = F.normalize(time_proj, dim=1, eps=eps)
    freq_proj = F.normalize(freq_proj, dim=1, eps=eps)

    # 1 - cosine similarity
    cosine_sim = F.cosine_similarity(time_proj, freq_proj, dim=1, eps=eps)
    loss = 1.0 - cosine_sim
    return loss.mean()


def tfc_total_loss(
    time_proj_1: torch.Tensor,
    time_proj_2: torch.Tensor,
    freq_proj_1: torch.Tensor,
    freq_proj_2: torch.Tensor,
    temperature: float = 0.2,
    lambda_tf: float = 0.5,
) -> dict[str, torch.Tensor]:
    """
    Compute the full TFC objective.

    Parameters
    ----------
    time_proj_1 : torch.Tensor
        Time-branch projection for augmented view 1, shape (B, D).
    time_proj_2 : torch.Tensor
        Time-branch projection for augmented view 2, shape (B, D).
    freq_proj_1 : torch.Tensor
        Frequency-branch projection for augmented view 1, shape (B, D).
    freq_proj_2 : torch.Tensor
        Frequency-branch projection for augmented view 2, shape (B, D).
    temperature : float, default=0.2
        Temperature parameter for NT-Xent.
    lambda_tf : float, default=0.5
        Weight controlling the balance between within-space contrastive losses
        and cross-space consistency loss.

    Returns
    -------
    dict[str, torch.Tensor]
        Dictionary containing:
        - "loss"
        - "loss_time"
        - "loss_freq"
        - "loss_consistency"
    """
    loss_time = nt_xent_loss(
        z1=time_proj_1,
        z2=time_proj_2,
        temperature=temperature,
    )

    loss_freq = nt_xent_loss(
        z1=freq_proj_1,
        z2=freq_proj_2,
        temperature=temperature,
    )

    loss_consistency_1 = time_frequency_consistency_loss(
        time_proj=time_proj_1,
        freq_proj=freq_proj_1,
    )

    loss_consistency_2 = time_frequency_consistency_loss(
        time_proj=time_proj_2,
        freq_proj=freq_proj_2,
    )

    loss_consistency = 0.5 * (loss_consistency_1 + loss_consistency_2)

    loss = lambda_tf * (loss_time + loss_freq) + (1.0 - lambda_tf) * loss_consistency

    return {
        "loss": loss,
        "loss_time": loss_time,
        "loss_freq": loss_freq,
        "loss_consistency": loss_consistency,
    }

In [ ]:
def build_time_views(x_time, padding_mask):
    x_time_view_1 = weak_time_augmentation(
        x=x_time,
        padding_mask=padding_mask,
        noise_std=0.01,
        feature_dropout_prob=0.05,
    )

    x_time_view_2 = weak_time_augmentation(
        x=x_time,
        padding_mask=padding_mask,
        noise_std=0.01,
        feature_dropout_prob=0.05,
    )

    return x_time_view_1, x_time_view_2

In [ ]:
def build_frequency_views(model, x_freq, padding_mask):
    fft_mag, fft_mask = model.compute_frequency_representation(
        x_freq=x_freq,
        padding_mask=padding_mask,
    )

    fft_view_1 = weak_frequency_augmentation(
        fft_mag=fft_mag,
        dropout_prob=0.05,
        noise_std=0.01,
    )

    fft_view_2 = weak_frequency_augmentation(
        fft_mag=fft_mag,
        dropout_prob=0.05,
        noise_std=0.01,
    )

    return fft_view_1, fft_view_2, fft_mask

In [ ]:
# one training loop
import torch


def tfc_train_step(
    model,
    batch,
    optimizer,
    device,
    temperature: float = 0.2,
    lambda_tf: float = 0.5,
):
    model.train()

    x_time = batch["x_time"].to(device)
    x_freq = batch["x_freq"].to(device)
    time_gaps = batch["time_gaps"].to(device)
    padding_mask = batch["padding_mask"].to(device)

    # ---- Build two time-domain views
    x_time_view_1, x_time_view_2 = build_time_views(
        x_time=x_time,
        padding_mask=padding_mask,
    )

    # ---- Build two frequency-domain views
    fft_view_1, fft_view_2, fft_mask = build_frequency_views(
        model=model,
        x_freq=x_freq,
        padding_mask=padding_mask,
    )

    # ---- Forward time branch
    time_pooled_1, time_proj_1 = model.forward_time_branch(
        x_time=x_time_view_1,
        time_gaps=time_gaps,
        padding_mask=padding_mask,
    )

    time_pooled_2, time_proj_2 = model.forward_time_branch(
        x_time=x_time_view_2,
        time_gaps=time_gaps,
        padding_mask=padding_mask,
    )

    # ---- Forward frequency branch
    freq_pooled_1, freq_proj_1 = model.forward_precomputed_frequency_branch(
        fft_mag=fft_view_1,
        fft_mask=fft_mask,
    )

    freq_pooled_2, freq_proj_2 = model.forward_precomputed_frequency_branch(
        fft_mag=fft_view_2,
        fft_mask=fft_mask,
    )

    # ---- Loss
    loss_dict = tfc_total_loss(
        time_proj_1=time_proj_1,
        time_proj_2=time_proj_2,
        freq_proj_1=freq_proj_1,
        freq_proj_2=freq_proj_2,
        temperature=temperature,
        lambda_tf=lambda_tf,
    )

    loss = loss_dict["loss"]

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    # ---- Useful diagnostics
    with torch.no_grad():
        time_proj_std = time_proj_1.std(dim=0).mean().item()
        freq_proj_std = freq_proj_1.std(dim=0).mean().item()
        time_proj_norm = time_proj_1.norm(dim=1).mean().item()
        freq_proj_norm = freq_proj_1.norm(dim=1).mean().item()

    metrics = {
        "loss": loss.item(),
        "loss_time": loss_dict["loss_time"].item(),
        "loss_freq": loss_dict["loss_freq"].item(),
        "loss_consistency": loss_dict["loss_consistency"].item(),
        "time_proj_std": time_proj_std,
        "freq_proj_std": freq_proj_std,
        "time_proj_norm": time_proj_norm,
        "freq_proj_norm": freq_proj_norm,
    }

    return metrics

In [ ]:
# validation step, no optimizer

@torch.no_grad()
def tfc_eval_step(
    model,
    batch,
    device,
    temperature: float = 0.2,
    lambda_tf: float = 0.5,
):
    model.eval()

    x_time = batch["x_time"].to(device)
    x_freq = batch["x_freq"].to(device)
    time_gaps = batch["time_gaps"].to(device)
    padding_mask = batch["padding_mask"].to(device)

    x_time_view_1, x_time_view_2 = build_time_views(
        x_time=x_time,
        padding_mask=padding_mask,
    )

    fft_view_1, fft_view_2, fft_mask = build_frequency_views(
        model=model,
        x_freq=x_freq,
        padding_mask=padding_mask,
    )

    _, time_proj_1 = model.forward_time_branch(
        x_time=x_time_view_1,
        time_gaps=time_gaps,
        padding_mask=padding_mask,
    )

    _, time_proj_2 = model.forward_time_branch(
        x_time=x_time_view_2,
        time_gaps=time_gaps,
        padding_mask=padding_mask,
    )

    _, freq_proj_1 = model.forward_precomputed_frequency_branch(
        fft_mag=fft_view_1,
        fft_mask=fft_mask,
    )

    _, freq_proj_2 = model.forward_precomputed_frequency_branch(
        fft_mag=fft_view_2,
        fft_mask=fft_mask,
    )

    loss_dict = tfc_total_loss(
        time_proj_1=time_proj_1,
        time_proj_2=time_proj_2,
        freq_proj_1=freq_proj_1,
        freq_proj_2=freq_proj_2,
        temperature=temperature,
        lambda_tf=lambda_tf,
    )

    metrics = {
        "loss": loss_dict["loss"].item(),
        "loss_time": loss_dict["loss_time"].item(),
        "loss_freq": loss_dict["loss_freq"].item(),
        "loss_consistency": loss_dict["loss_consistency"].item(),
        "time_proj_std": time_proj_1.std(dim=0).mean().item(),
        "freq_proj_std": freq_proj_1.std(dim=0).mean().item(),
        "time_proj_norm": time_proj_1.norm(dim=1).mean().item(),
        "freq_proj_norm": freq_proj_1.norm(dim=1).mean().item(),
    }

    return metrics

In [ ]:
# epoch runner

def run_tfc_epoch(
    model,
    loader,
    optimizer,
    device,
    train: bool = True,
    temperature: float = 0.2,
    lambda_tf: float = 0.5,
):
    all_metrics = []

    for batch in loader:
        if train:
            metrics = tfc_train_step(
                model=model,
                batch=batch,
                optimizer=optimizer,
                device=device,
                temperature=temperature,
                lambda_tf=lambda_tf,
            )
        else:
            metrics = tfc_eval_step(
                model=model,
                batch=batch,
                device=device,
                temperature=temperature,
                lambda_tf=lambda_tf,
            )

        all_metrics.append(metrics)

    summary = {}
    metric_names = all_metrics[0].keys()

    for name in metric_names:
        summary[name] = sum(m[name] for m in all_metrics) / len(all_metrics)

    return summary

In [ ]:
# full training loop

import copy
from ssl_tfc_model import TFCSequenceModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = TFCSequenceModel(
    time_input_dim=105,
    freq_input_dim=105,
    time_hidden_dim=128,
    freq_hidden_dim=128,
    projection_dim=64,
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4,
)

num_epochs = 20
temperature = 0.2
lambda_tf = 0.5

best_val_loss = float("inf")
best_state_dict = None
history = []

for epoch in range(1, num_epochs + 1):
    train_metrics = run_tfc_epoch(
        model=model,
        loader=train_loader_tfc,
        optimizer=optimizer,
        device=device,
        train=True,
        temperature=temperature,
        lambda_tf=lambda_tf,
    )

    val_metrics = run_tfc_epoch(
        model=model,
        loader=val_loader_tfc,
        optimizer=None,
        device=device,
        train=False,
        temperature=temperature,
        lambda_tf=lambda_tf,
    )

    epoch_record = {
        "epoch": epoch,
        **{f"train_{k}": v for k, v in train_metrics.items()},
        **{f"val_{k}": v for k, v in val_metrics.items()},
    }
    history.append(epoch_record)

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_metrics['loss']:.4f} | "
        f"val_loss={val_metrics['loss']:.4f} | "
        f"train_time={train_metrics['loss_time']:.4f} | "
        f"train_freq={train_metrics['loss_freq']:.4f} | "
        f"train_cons={train_metrics['loss_consistency']:.4f} | "
        f"time_std={train_metrics['time_proj_std']:.4f} | "
        f"freq_std={train_metrics['freq_proj_std']:.4f}"
    )

    if val_metrics["loss"] < best_val_loss:
        best_val_loss = val_metrics["loss"]
        best_state_dict = copy.deepcopy(model.state_dict())

In [ ]:
# save best result
if best_state_dict is not None:
    model.load_state_dict(best_state_dict)
tfc_save_path = MODELS_DIR / "tfc_pretrained_model_B3.pt"
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "best_val_loss": best_val_loss,
        "history": history,
        "time_input_dim": 105,
        "freq_input_dim": 105,
        "time_hidden_dim": 128,
        "freq_hidden_dim": 128,
        "projection_dim": 64,
    },tfc_save_path
)

print("Saved best TFC model.")